# FaultLines: reproduce the paper's tables
This notebook rebuilds the main numbers of *Fault Lines: Where Local Gemma 4 Coding Agents Break* (Gemma 4 Developer Agent, Paper Track) from the public dataset **FaultLines-Traces Labels**. It needs no GPU and no competition data: everything comes from the labels and step metadata of 242 runs.

Experiments: **E1** pilot (16 tasks), **E4** prompt-only fix on the same 16 (RQ4), **E3** the 70 judgeable tasks, **E5** E3 with `max_output_tokens` 16,384 → 8,192, **E6** an identical repeat of E5.

In [ ]:
import glob, json, math, os
import pandas as pd

hits = sorted(glob.glob('/kaggle/input/**/runs_summary.csv', recursive=True))
assert hits, 'Attach the dataset nawazs4265/faultlines-traces-labels'
D = os.path.dirname(hits[0])
runs = pd.read_csv(os.path.join(D, 'runs_summary.csv'))
env = pd.read_csv(os.path.join(D, 'envcheck.csv'))
diff = pd.read_csv(os.path.join(D, 'task_difficulty.csv'))
steps = {}
for p in sorted(glob.glob(os.path.join(D, '*_runs.jsonl'))):
    for line in open(p):
        r = json.loads(line)
        steps[(r['experiment'], r['task_id'])] = r['steps']
runs['submitted'] = runs['exit_reason'].eq('submitted')
runs['resolved'] = runs['resolved'].astype(bool)
NAMES = {'e1_pilot': 'E1 pilot', 'e4_rq4': 'E4 prompt fix', 'e3_judgeable70': 'E3', 'e5_reserve8192': 'E5 reserve 8,192', 'e6_reserve8192_repeat': 'E6 repeat of E5'}

def wilson(x, n, z=1.96):
    if n == 0:
        return (float('nan'), float('nan'))
    p = x / n; d = 1 + z * z / n; c = p + z * z / (2 * n)
    s = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (100 * (c - s) / d, 100 * (c + s) / d)

def mcnemar(a, b):
    """Exact two-sided McNemar test on paired booleans a -> b."""
    gain = int(((~a) & b).sum()); loss = int((a & (~b)).sum()); n = gain + loss
    if n == 0:
        return gain, loss, 1.0
    p = 2 * sum(math.comb(n, i) for i in range(min(gain, loss) + 1)) / 2 ** n
    return gain, loss, round(min(1.0, p), 3)

def paired(e1, e2, col):
    a = runs[runs.experiment == e1].set_index('task_id')[col]
    b = runs[runs.experiment == e2].set_index('task_id')[col].reindex(a.index)
    return mcnemar(a.astype(bool), b.astype(bool))

def step_stats(e):
    st = [s for (x, _), ss in steps.items() if x == e for s in ss]
    ed = [s for s in st if s['tool_name'] == 'edit_file']
    return {'turns': len(st),
            'schema-invalid calls': sum(s['call_status'] in ('rejected', 'repaired') for s in st),
            'edit_file calls': len(ed),
            'edit_file failed': sum(bool(s['runtime_error']) or s['call_status'] == 'rejected' for s in ed),
            'edit_file no match': sum(s['runtime_error'] == 'EDIT_NO_MATCH' for s in ed)}

print(D); print(runs.experiment.value_counts().to_string())

## Overview: resolution by experiment (Wilson 95% intervals)

In [ ]:
rows = []
for e, g in runs.groupby('experiment', sort=False):
    x, n = int(g.resolved.sum()), len(g)
    lo, hi = wilson(x, n)
    rows.append({'experiment': NAMES.get(e, e), 'runs': n, 'resolved': x, 'rate %': round(100 * x / n, 1), 'CI %': f'{lo:.0f}-{hi:.0f}',
                 'submitted': int(g.submitted.sum()), 'context crashes': int(g.harness_stop.eq('context_window_exceeded').sum()),
                 'schema-invalid turns %': round(100 * step_stats(e)['schema-invalid calls'] / step_stats(e)['turns'], 1), 'minutes/task': round(g.wall_s.mean() / 60, 1)})
pd.DataFrame(rows)

## Section 7.2-7.3: pilot (E1), RQ1 and RQ2
Adjudicated labels, schema-invalid turns and `edit_file` failures in the 16-task pilot.

Definitions used throughout: a *schema-invalid* turn is one whose call toolguard rejected or repaired; an `edit_file` call *failed* if it was rejected or returned an error.

In [ ]:
e1 = runs[runs.experiment == 'e1_pilot']
print('adjudicated labels:'); print(e1.human_label.value_counts().to_string())
ss = step_stats('e1_pilot')
print(f"\nschema-invalid calls (rejected or repaired by toolguard): {ss['schema-invalid calls']} of {ss['turns']} turns")
print(f"edit_file failed (error or rejected): {ss['edit_file failed']} of {ss['edit_file calls']}, of which 'old_string not found': {ss['edit_file no match']}")
agree = (e1.auto_label == e1.human_label).sum()
print(f"auto-labeler (v0.2 as stored) agrees with adjudication on {agree} of {len(e1)} runs")

## Section 7.4: environment validity
A task is judgeable if its reference patch passes and the empty patch fails in the starter sandbox.

In [ ]:
env['judgeable'] = env['judgeable'].astype(str).str.lower().isin(['1', 'true'])
t = env.groupby('repo').agg(tasks=('instance_id', 'size'), judgeable=('judgeable', 'sum'))
t.loc['all'] = t.sum()
t

## Section 7.6: RQ4, a prompt-only fix (E1 vs E4, same 16 tasks)

In [ ]:
def arm(e):
    g = runs[runs.experiment == e]; ss = step_stats(e)
    return {'resolved': int(g.resolved.sum()), 'called submit_patch': int(g.submitted.sum()),
            'edit_file failed': f"{ss['edit_file failed']}/{ss['edit_file calls']}",
            'schema-invalid turns': f"{ss['schema-invalid calls']}/{ss['turns']}", 'minutes/task': round(g.wall_s.mean() / 60, 1)}
print(pd.DataFrame({'before (E1)': arm('e1_pilot'), 'after (E4)': arm('e4_rq4')}).to_string())
print('\nsubmissions E1 -> E4 (gained, lost, exact McNemar p):', paired('e1_pilot', 'e4_rq4', 'submitted'))
print('resolution  E1 -> E4:', paired('e1_pilot', 'e4_rq4', 'resolved'))

## Section 7.7: E3 on the 70 judgeable tasks
Primary labels (adjudicated where available), the context-window crash, and completion lengths.

The paper's completion statistics (median 84 tokens over 2,177 calls) come from the full traces; here 38 turns have no per-step token count, so the median comes out slightly different.

In [ ]:
e3 = runs[runs.experiment == 'e3_judgeable70']
lab = e3.human_label.fillna(e3.auto_label)
print(lab.value_counts().to_string())
print('\ncontext-window crashes:', int(e3.harness_stop.eq('context_window_exceeded').sum()))
comp = pd.Series([s['completion_tokens'] for (e, _), st in steps.items() if e == 'e3_judgeable70' for s in st if s.get('completion_tokens') is not None])
print(f"completions: n={len(comp)}, median={comp.median():.0f}, p99={comp.quantile(0.99):.0f}, >6144: {(comp > 6144).sum()}")
print(e3.groupby('repo').resolved.agg(['sum', 'size']).to_string())
ss = step_stats('e3_judgeable70'); print(f"edit_file failed: {ss['edit_file failed']} of {ss['edit_file calls']}")

## Section 7.8: E5 (output reserve halved) and E6 (identical repeat)

In [ ]:
for a, b in [('e3_judgeable70', 'e5_reserve8192'), ('e5_reserve8192', 'e6_reserve8192_repeat'), ('e3_judgeable70', 'e6_reserve8192_repeat')]:
    print(f"{NAMES[a]} -> {NAMES[b]}: resolved (gained, lost, p) = {paired(a, b, 'resolved')}; submitted = {paired(a, b, 'submitted')}")
e5p = runs[runs.experiment == 'e5_reserve8192']
print('\nE5 runs with prompts above 16,384 tokens:', int((e5p.max_prompt_tokens > 16384).sum()), '| max prompt:', int(e5p.max_prompt_tokens.max()))

## Run-to-run variance: how often each judgeable task is resolved across E3, E5 and E6

In [ ]:
print(diff.times_resolved.value_counts().sort_index().rename('tasks').to_string())
print(f"\nresolved in all three: {(diff.times_resolved == 3).sum()}, in at least one: {(diff.times_resolved > 0).sum()}, never: {(diff.times_resolved == 0).sum()}")
diff.sort_values(['times_resolved', 'task_id'], ascending=[False, True]).head(20)